# Import

In [ ]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import pickle
import math
import igraph as ig
import seaborn as sns
import statistics
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from matplotlib.patches import Patch
import matplotlib.cm as cm

# Initialization

In [ ]:
disease = input("Disease: ")

In [ ]:
def load_info(disease):
    DISEASE_FOLDER = f"../output/{disease}/"
    RESULT_FOLDER = DISEASE_FOLDER + "leiden_results/"
    DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{disease}/"
    RESULT_COMMUNITIES = "result_communities"
    RESULT_GRAPH = "result_graph"

    with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
    with open(DGIDB_DIRECTORY + f"drug_to_index.json", "r") as file:
        DGIDB_drug_to_index = json.load(file)
    # Loading result graph and communities
    with open(f"{RESULT_FOLDER}/{RESULT_COMMUNITIES}.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
        communities_selected = pickle.load(f)
    with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
        graph = pickle.load(f)
    
    DGIDB_genes = set(DGIDB_gene_to_index.keys())
    DGIDB_drugs = set(DGIDB_drug_to_index.keys())
    index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}
    
    return {"communities": communities,
            "communities_selected": communities_selected,
            "graph": graph,
            "gene_to_index_distinct": gene_to_index_distinct,
            "index_to_gene_distinct": index_to_gene_distinct,
            "DGIDB_gene_to_index": DGIDB_gene_to_index,
            "DGIDB_drug_to_index": DGIDB_drug_to_index,
            "DGIDB_genes": DGIDB_genes,
            "DGIDB_drugs": DGIDB_drugs
            }

In [ ]:
disease_info = load_info(disease)

## ncbi

In [ ]:
def index_to_ncbi(comms,index_to_ncbi):
    comms_ncbi = [list(map(index_to_ncbi.get, c)) for c in comms]
    return comms_ncbi

def DGIDB_count(c,DGIDB_genes_ncbi):
    return set(c) & set(DGIDB_genes_ncbi)

disease_info['communities_ncbi'] = index_to_ncbi(disease_info['communities'],
                                                        disease_info['index_to_gene_distinct'])
disease_info['communities_selected_ncbi'] = index_to_ncbi(disease_info['communities_selected'],
                                                    disease_info['index_to_gene_distinct'])

# Centrality Analysis

## prep

In [ ]:
def is_dgidb(idx):
    gene = disease_info['index_to_gene_distinct'][idx]
    return gene in disease_info['DGIDB_genes']

In [ ]:
def igraph_from_sparse(M, directed=False, weighted=True):
    """
    Build an igraph Graph from a SciPy sparse matrix.

    Parameters
    ----------
    M : scipy.sparse matrix (CSR/CSC/COO)
        Adjacency or weighted adjacency matrix.
    directed : bool
        Whether the graph is directed.
    weighted : bool
        Whether to store values as edge weights.

    Returns
    -------
    ig.Graph
    """
    M = M.tocoo()

    edges = list(zip(M.row.tolist(), M.col.tolist()))
    g = ig.Graph(edges=edges, directed=directed)

    if weighted:
        g.es["weight"] = M.data.tolist()

    return g

# Community preprocessing
def nx_to_igraph(G: nx.Graph, weight: str | None = "weight") -> ig.Graph:
    """
    Convert a NetworkX graph to an iGraph graph.
    
    Parameters
    ----------
    G : nx.Graph or nx.DiGraph
        Your NetworkX graph.
    weight : str or None, optional
        Name of the edge attribute to treat as weight.
        If None, graph is treated as unweighted.

    Returns
    -------
    ig.Graph
        An iGraph object with:
        - g.vs['name'] = node labels
        - g.es['weight'] = weights (if provided)
    """
    # 1) Keep original node labels
    nodes = list(G.nodes())
    idx_map = {node: i for i, node in enumerate(nodes)}

    # 2) Convert edges
    edges = [(idx_map[u], idx_map[v]) for u, v in G.edges()]

    # 3) Initialize iGraph
    g_ig = ig.Graph(edges=edges, directed=G.is_directed())
    g_ig.vs["name"] = nodes

    # 4) Add weights if available
    if weight is not None:
        # Extract weights or default to 1.0
        weights = [G[u][v].get(weight) for u, v in G.edges()]
        g_ig.es["weight"] = weights

    return g_ig

def intramodule_closeness(G, community_nodes, weight="weight"):
    """
    Intramodule closeness centrality using the induced subgraph of a community.
    
    Parameters
    ----------
    G : nx.Graph
        Original graph.
    community : iterable
        Nodes in the community (subset of G).
    weight : str or None
        Edge attribute to use as weights.
    
    Returns
    -------
    dict
        Mapping node -> intramodule closeness centrality.
    """
    # Induced subgraph
    sub = G.subgraph(community_nodes)

    # Regular closeness, but only inside H
    cl = sub.closeness(weights=weight, normalized=True)
    closeness = {sub.vs[i]["name"]: float(cl[i]) for i in range(sub.vcount())}
    return closeness

def degree_for_community(G, community_nodes, weight="weight"):
    """
    Intramodule closeness centrality using the induced subgraph of a community.
    
    Parameters
    ----------
    G : nx.Graph
        Original graph.
    community : iterable
        Nodes in the community (subset of G).
    weight : str or None
        Edge attribute to use as weights.
    
    Returns
    -------
    dict
        Mapping node -> intramodule closeness centrality.
    """
    # Induced subgraph
    sub = G.subgraph(community_nodes)

    # Regular closeness, but only inside H
    cl = sub.strength(weights=weight)
    strength = {sub.vs[i]["name"]: float(cl[i]) for i in range(sub.vcount())}
    return strength

def pagerank_for_community(G, community_nodes, weight="weight"):
    # 1. Build subgraph by vertex names
    sub = G.subgraph(community_nodes).copy()

    # 2. Compute PageRank
    pr = nx.pagerank(sub, weight=weight)

    return pr

def eig_centrality_for_community(G, community_nodes, weight="weight"):
    # 1. Build subgraph by vertex names
    sub = G.subgraph(community_nodes).copy()

    # 2. Compute PageRank
    pr = nx.eigenvector_centrality(sub, weight=weight, max_iter=5000, tol=1e-6)

    return pr

def betweenness_for_community(
    g: ig.Graph,
    community_nodes,
    weight: str | None = "weight"
):
    """
    Compute betweenness centrality for nodes inside a community.

    Parameters
    ----------
    g : ig.Graph
        Full igraph graph (must have vs['name']).
    community_nodes : list of str
        Node names belonging to this community.
    weight : str or None
        Edge weight attribute name. None = unweighted betweenness.
    normalized : bool
        Normalize betweenness by maximum possible value.

    Returns
    -------
    dict {node_name : betweenness_score}
    """

    # Build subgraph of the community
    sub = g.subgraph(community_nodes)
    sub.vs["orig_vid"] = community_nodes
    n = sub.vcount()

    w = np.asarray(sub.es[weight], dtype=float)

    # Rescale so the smallest weight becomes 1.0 (avoids epsilon issues)
    w = w / w.min()

    bt = sub.betweenness(weights=w.tolist())  # pass list is most robust

    # Freeman normalization (undirected)
    # norm = (n - 1) * (n - 2) / 2
    # bt = [b / norm for b in bt]

    return {sub.vs[i]["orig_vid"]: bt[i] for i in range(n)}

## create graphs

In [ ]:
def knn_from_similarity(S: csr_matrix, k: int) -> csr_matrix:
    S = S.tocsr()
    rows, cols, data = [], [], []

    for i in range(S.shape[0]):
        start, end = S.indptr[i], S.indptr[i + 1]
        row_data = S.data[start:end]
        row_cols = S.indices[start:end]

        if len(row_data) > k:
            idx = np.argpartition(row_data, -k)[-k:]
            rows.extend([i] * k)
            cols.extend(row_cols[idx])
            data.extend(row_data[idx])
        else:
            rows.extend([i] * len(row_data))
            cols.extend(row_cols)
            data.extend(row_data)

    return csr_matrix((data, (rows, cols)), shape=S.shape)

In [ ]:
with open(f"../output/msigdb_adjacency_matrix.pkl", "rb") as f:
    msigdb_adjacency_matrix = pickle.load(f)


In [ ]:
with open(f"../output/{disease}/leiden_results/ddm_agg.pkl", "rb") as f:
    ddm_agg = pickle.load(f)

In [ ]:
ddm_agg_igraph = igraph_from_sparse(ddm_agg)

In [ ]:
similarity_nxgraph = disease_info["graph"]
distance_nxgraph = similarity_nxgraph.copy()
    
eps = 1e-12
for _, _, d in distance_nxgraph.edges(data=True):
    s = d["weight"]
    if (s == 1):
        s -= eps
    d["weight"] = -math.log(max(s, eps))

In [ ]:
similarity_igraph = nx_to_igraph(similarity_nxgraph)

In [ ]:
distance_igraph = nx_to_igraph(distance_nxgraph)

In [ ]:
avg_weight = sum(distance_igraph.es["weight"]) / distance_igraph.ecount()
print("Average edge weight:", avg_weight)

## centrality calculation

### Betweenness Scores (very time consuming)

In [ ]:
w = np.array(distance_igraph.es["weight"], dtype=float)

print("Any NaN? ", np.isnan(w).any())
print("Any inf? ", np.isinf(w).any())
print("Min weight:", np.nanmin(w))
print("Count <= 0:", np.sum(w <= 0))

In [ ]:
all_betweenness_scores = []
for community in disease_info['communities'][:len(disease_info['communities_selected'])]:
    betweenness_scores = betweenness_for_community(ddm_agg_igraph,community)
    all_betweenness_scores.append(betweenness_scores)
    print(betweenness_scores)

In [ ]:
# combine all scores
betweenness_scores_full = {}
for s in all_betweenness_scores:
    betweenness_scores_full.update(s)

In [ ]:
top_10_betweenness_scores = dict(sorted(betweenness_scores_full.items(), key=lambda item: item[1], reverse=True)[:10])

In [ ]:
top_10_betweenness_scores_dgidb = {k: v for k, v in top_10_betweenness_scores.items() if is_dgidb(k)}

In [ ]:
# all_betweenness_scores_zscore = []
# for bs in all_betweenness_scores:
#     vals = np.array(list(bs.values()), dtype=float)
#     z = (vals - vals.mean()) / vals.std()

#     bs_z = dict(zip(bs.keys(), z))
#     all_betweenness_scores_zscore.append(bs_z)

In [ ]:
# all_betweenness_scores_zscore

#### Save the Resulting Dictionary

In [ ]:
# Save the resulting dictionary
with open(f"../output/{disease}/all_betweenness_scores.json", "w") as f:
    json.dump(all_betweenness_scores, f, indent=2)

### Eigenvector Centrality

In [ ]:
all_eig_scores = []
for community in disease_info['communities'][:len(disease_info['communities_selected'])]:
    eig_scores = eig_centrality_for_community(similarity_nxgraph,community)
    all_eig_scores.append(eig_scores)
    print(eig_scores)

### degree centrality

In [ ]:
all_degree_scores = []
for community in disease_info['communities'][:len(disease_info['communities_selected'])]:
    degree_scores = degree_for_community(similarity_igraph,community)
    all_degree_scores.append(degree_scores)
    print(degree_scores)

### closeness centrality

In [ ]:
all_closeness_scores = []
for community in disease_info['communities'][:len(disease_info['communities_selected'])]:
    closeness_scores = intramodule_closeness(distance_igraph,community)
    all_closeness_scores.append(closeness_scores)
    print(closeness_scores)

# DGIDB genes betweenness score

In [ ]:
def DGIDB_score(all_scores):
    # DGIDB_betweenness_mat = []
    # DGIDB_comm_idx = []
    DGIDB_genes = disease_info['DGIDB_genes']
    comms_ncbi = disease_info['communities_ncbi']
    gtid = disease_info['gene_to_index_distinct']
    num_selected_comm = len(disease_info['communities_selected'])

    print(f"\033[1m{disease}: \033[0m")
    for i,(c,s) in enumerate(zip(comms_ncbi[:num_selected_comm],all_scores[:num_selected_comm])):
        scores = [val for _,val in s.items()]
        dgidb_genes = list(DGIDB_count(c,DGIDB_genes))
        dgidb_scores = [s[gtid[g]] for g in dgidb_genes]
        # dgidb_percentiles = [sum(v <= x for _,v in s.items()) / len(s) for x in dgidb_scores]    
        if len(dgidb_scores) == 0:
            continue
        # DGIDB_comm_idx.append(i)
        print(f"Community {i}:")
        
        # average without outliers
        x = np.array(scores)

        q1 = np.percentile(x, 25)
        q3 = np.percentile(x, 75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        filtered = x[(x >= lower) & (x <= upper)]
        mean_no_outliers = filtered.mean()
        
        print(f"Average score (without outliers): {mean_no_outliers}")
        
        # median all
        median_score = statistics.median([val for _,val in s.items()])
        print(f"Median score: {median_score}")
        
        # average dgidb
        dgidb_avg_score = sum(dgidb_scores) / len(dgidb_scores)
        print(f"DGIDB Average score: {dgidb_avg_score}")
    
        dgidb_scores_sorted = sorted(dgidb_scores, reverse=True)
        # dgidb_scores_sorted_padded = dgidb_scores_sorted + [0] * max(0, 10 - len(dgidb_scores_sorted))
        # DGIDB_betweenness_mat.append(dgidb_scores_sorted_padded[:10])
        print(f"Scores: {dgidb_scores_sorted}")
        print()    

In [ ]:
len(disease_info["communities_ncbi"])

In [ ]:
DGIDB_score(all_betweenness_scores)

In [ ]:
DGIDB_score(all_eig_scores)

In [ ]:
DGIDB_score(all_degree_scores)

## heatmap

In [ ]:
# DGIDB_betweenness_mat = np.array(DGIDB_betweenness_mat)

In [ ]:
# DGIDB_comm_idx

In [ ]:
# plt.figure(figsize=(6, 4))
# ax = sns.heatmap(
#     DGIDB_betweenness_mat,
#     mask=(DGIDB_betweenness_mat == 0),
#     cmap="viridis",
#     annot=True,      # set True to show values
#     cbar=True,
# )

# ax.set_yticklabels(DGIDB_comm_idx)
# ax.set_ylabel("Community Index")
# ax.set_xlabel("Top 10 DGIDB genes by betweenness score")
# plt.tight_layout()
# plt.savefig(f"../../Graphs/{disease}/DGIDB_betweenness_score_heatmap.png")
# plt.show()